# Encoder–HistGB: Transformer optimization

The baseline remains the unchanged pipeline from **features_selection.ipynb**.
HistGB receives its existing features plus frozen encoder embeddings; only the encoder's
architecture and training hyperparameters are searched. Inputs remain original imputed
features, missingness indicators and elapsed time, with bidirectional attention and no CLS token.

1. **Search:** 40 Optuna trials, three patient splits, seed 42.
2. **Confirm:** best three configurations, five patient splits, seeds 42, 43 and 44.
3. **Final model:** winning configuration, all training patients, a fixed seed.

Every fit keeps the internal 20% patient holdout for epoch selection, followed by a fresh
encoder fit on all fold-training patients. The objective is the complete Encoder–HistGB
validation MSE. There are no out-of-fold embeddings.

## 1. Reuse the exact baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Paths work from the project root or either notebook folder. N_TRIALS and TOP_K control the search and confirmation budgets.

In [ ]:
import json
import os
import optuna
from pathlib import Path
from copy import deepcopy
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)  # The reference setup reads data relative to the project.
    for i in [1, 3, 4, 6, 8]:  # Imports/data, history, imputation, features, pipeline.
        exec("".join(reference["cells"][i]["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)  # Imputer helpers creation + one-hot → predictive imputation/flags → engineered features → median imputation/flags → no selection → tuned HistGB (unfitted).

ENCODER = dict(width=64, heads=4, layers=2, dropout=0.1,
               ff_multiplier=2, mlp_layers=0, mlp_units=32) # ff_multiplier sets the hidden width of the feed-forward network inside each Transformer block. With embedding width 64 and ff_multiplier = 3, it becomes 64 → 192 → 64. The output embedding still has 64 values.
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 100, 10, 128
LEARNING_RATE, WEIGHT_DECAY = 1e-3, 1e-4
SEEDS = [42, 43, 44]
N_TRIALS, TOP_K = 40, 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=1.8115270802885095,
                              learning_rate=0.042592090635273853,
                              max_features=0.6078232265551354, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=25,
                              random_state=42)


## 2. Build patients sequences (Encoder dataset)

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [ ]:
def encoder_features(imputed):
    """Prepare per-visit inputs without patient IDs, adding elapsed time.

    Keep the original imputed features and missingness indicators. Compute
    elapsed years as age minus the patient's age at their first available visit.
    Patient IDs are used only for this grouping, not as encoder inputs.
    """
    
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    """Build one padded sequence per patient, with visits ordered by age.

    Features and targets must follow the same row order as raw. Return tensors
    of features [patients, visits, features], targets [patients, visits], and
    original row positions [patients, visits]. Padding positions are marked -1
    so attention and loss can ignore them; real positions restore embeddings
    to the original table order. Without targets, zero labels are unused
    placeholders for prediction.
    """
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Parameterized bidirectional encoder

Dense input projection → TransformerEncoder layers → one embedding per visit.
Optuna varies width, heads, layers, feed-forward size and dropout. The regression head can
contain zero, one or two hidden layers; it supervises training but is not fed to HistGB.
Embedding width remains the Transformer width regardless of the regression head architecture.

In [ ]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1,
                 ff_multiplier=2, mlp_layers=0, mlp_units=32):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=ff_multiplier * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # 1. Initialize the encoder’s weights
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        # 2. Build the regression head
        head, input_dim = [], width
        for _ in range(mlp_layers):
            head.extend([nn.Linear(input_dim, mlp_units), nn.GELU(), nn.Dropout(dropout)])
            input_dim = mlp_units
        self.head = nn.Sequential(*head, nn.Linear(input_dim, 1))

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Train the encoder inside a training split

Both input and target scaling are fitted on training patients only. Target scaling improves
optimization; reported predictions and MSE are converted back to OFF score units.
Early stopping uses its own patient holdout, including independently fitted preprocessing.
The selected epoch count is then used to refit a fresh encoder on all outer training patients.

In [10]:
def train_network(training, seed, epochs, validation=None, params=None):
    torch.manual_seed(seed)
    params = {**ENCODER, "learning_rate": LEARNING_RATE,
              "weight_decay": WEIGHT_DECAY, **(params or {})}
    network = VisitEncoder(training.tensors[0].shape[-1],
                           **{key: params[key] for key in ENCODER}).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=params["learning_rate"],
                                 weight_decay=params["weight_decay"])
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, imputed, seed, params=None):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping, params=params)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh model and scaling, now fitted on every patient in the outer training split.
    features = encoder_features(imputed)
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, _ = train_network(training, seed, epochs, params=params)
    network.requires_grad_(False)
    return dict(network=network, scaler=scaler, target_mean=mean, target_scale=scale,
                epochs=epochs, history=history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; HistGB receives only the contextual
embeddings, with no CLS token, pooling or predicted-score column.

In [11]:
@torch.no_grad()
def encoder_outputs(fitted, raw, imputed):
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.projection.out_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6. Prepare the five patient splits once

The unchanged preprocessing and HistGB baseline are fitted once per split, then reused across
configurations. Search uses the first three splits; confirmation uses all five. Encoder early
stopping and refitting remain inside each fold and seed. No embeddings cross fold boundaries.

In [12]:
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
prepared_folds = []
with threadpool_limits(limits=3):
    for train_idx, val_idx in cv.split(X, y, groups=X["patient_id"]):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        train = prep.fit_transform(raw_train, yt)
        validation = prep.transform(raw_val)
        original = clone(baseline[-1]).fit(train, yt)
        prepared_folds.append(dict(
            raw_train=raw_train, raw_val=raw_val, yt=yt, yv=yv,
            train=train, validation=validation,
            imputed_train=prep[:2].transform(raw_train), imputed_val=prep[:2].transform(raw_val),
            baseline_mse=mean_squared_error(yv, original.predict(validation)),
        ))


def evaluate_config(params, folds, seeds, trial=None):
    scores = []
    with threadpool_limits(limits=3):
        for fold_number, fold in enumerate(folds, 1):
            for seed in seeds:
                fitted = fit_encoder(fold["raw_train"], fold["yt"], fold["imputed_train"], seed, params)
                z_train, _ = encoder_outputs(fitted, fold["raw_train"], fold["imputed_train"])
                z_val, prediction = encoder_outputs(fitted, fold["raw_val"], fold["imputed_val"])
                model = clone(baseline[-1]).fit(fold["train"].join(z_train), fold["yt"])
                mse = mean_squared_error(fold["yv"], model.predict(fold["validation"].join(z_val)))
                scores.append(dict(fold=fold_number, seed=seed, epochs=fitted["epochs"],
                                   HistGB=fold["baseline_mse"], Encoder_HistGB=mse,
                                   Transformer=mean_squared_error(fold["yv"], prediction),
                                   delta_MSE=mse - fold["baseline_mse"]))
                if trial is None:
                    print(f"Fold {fold_number}/{len(folds)}, seed {seed}: MSE {mse:.4f}")
            if trial is not None:
                trial.report(np.mean([s["Encoder_HistGB"] for s in scores]), step=fold_number)
                if trial.should_prune():
                    raise optuna.TrialPruned()
    return pd.DataFrame(scores)

## 7. Optuna search: three splits, seed 42

Search 40 configurations (adjust N_TRIALS to 30–50). The current configuration is evaluated
first as a reference. The objective is mean **Encoder–HistGB MSE**, not standalone encoder MSE.
Unpromising trials can be pruned after at least two completed folds.

Search space: width 32/40/64; heads 2/4/8 (all divide these widths); 1–3 encoder layers;
feed-forward width 2×/3×/4×; dropout 0–0.3; 0–2 regression-head hidden layers with
16/32/64 units; learning rate 1e-4–3e-3 and weight decay 1e-6–1e-2, both on logarithmic scales.

In [13]:
def suggest_params(trial):
    params = dict(
        width=trial.suggest_categorical("width", [32, 40, 64]),
        heads=trial.suggest_categorical("heads", [2, 4, 8]),
        layers=trial.suggest_int("layers", 1, 3),
        ff_multiplier=trial.suggest_int("ff_multiplier", 2, 4),
        dropout=trial.suggest_float("dropout", 0.0, 0.3),
        mlp_layers=trial.suggest_int("mlp_layers", 0, 2),
        learning_rate=trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
    )
    if params["mlp_layers"]:
        params["mlp_units"] = trial.suggest_categorical("mlp_units", [16, 32, 64])
    return params


def objective(trial):
    scores = evaluate_config(suggest_params(trial), prepared_folds[:3], [42], trial)
    trial.set_user_attr("epochs", scores["epochs"].tolist())
    return scores["Encoder_HistGB"].mean()


study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
initial_params = {**ENCODER, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY}
if initial_params["mlp_layers"] == 0:
    initial_params.pop("mlp_units")
study.enqueue_trial(initial_params)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
study.trials_dataframe().sort_values("value")

[I 2026-11-01 17:04:36,374] A new study created in memory with name: no-name-09dde26e-6843-4b25-9919-1dc66c6de388
Best trial: 0. Best value: 8.2774:   2%|▎         | 1/40 [02:36<1:41:51, 156.71s/it]

[I 2026-11-01 17:07:13,022] Trial 0 finished with value: 8.27739777758965 and parameters: {'width': 64, 'heads': 4, 'layers': 2, 'ff_multiplier': 2, 'dropout': 0.1, 'mlp_layers': 0, 'learning_rate': 0.001, 'weight_decay': 0.0001}. Best is trial 0 with value: 8.27739777758965.


Best trial: 0. Best value: 8.2774:   5%|▌         | 2/40 [05:14<1:39:36, 157.29s/it]

[I 2026-11-01 17:09:50,664] Trial 1 finished with value: 10.192713318276448 and parameters: {'width': 40, 'heads': 2, 'layers': 1, 'ff_multiplier': 4, 'dropout': 0.18033450352296262, 'mlp_layers': 2, 'learning_rate': 0.00010725209743172001, 'weight_decay': 0.007579479953348004, 'mlp_units': 16}. Best is trial 0 with value: 8.27739777758965.


Best trial: 0. Best value: 8.2774:   8%|▊         | 3/40 [06:35<1:15:37, 122.65s/it]

[I 2026-11-01 17:11:12,129] Trial 2 finished with value: 9.161602381971159 and parameters: {'width': 64, 'heads': 8, 'layers': 1, 'ff_multiplier': 2, 'dropout': 0.1099085529881075, 'mlp_layers': 1, 'learning_rate': 0.0014447746112718689, 'weight_decay': 6.290644294586156e-06, 'mlp_units': 32}. Best is trial 0 with value: 8.27739777758965.


... (71 outputs left out)

[I 2026-11-01 18:45:53,372] Trial 38 pruned. 


Best trial: 19. Best value: 7.34163: 100%|██████████| 40/40 [1:44:03<00:00, 156.10s/it]

[I 2026-11-01 18:48:40,065] Trial 39 pruned. 


,number,value,datetime_start,datetime_complete,duration,params_dropout,params_ff_multiplier,params_heads,params_layers,params_learning_rate,params_mlp_layers,params_mlp_units,params_weight_decay,params_width,user_attrs_epochs,system_attrs_fixed_params,system_attrs_tpe:relative_params:0,state
19,19,7.341625,2026-11-01 17:51:10.667609,2026-11-01 17:54:48.057806,0 days 00:03:37.390197,0.134789,3,8,3,0.001759,2,16.0,0.000044,32,"[56, 67, 86]",NaN,"{""dropout"": 0.1347886862599206, ""ff_multiplier...",COMPLETE
27,27,7.348857,2026-11-01 18:14:41.039320,2026-11-01 18:17:03.262763,0 days 00:02:22.223443,0.072177,3,8,3,0.002732,2,32.0,0.000056,32,"[20, 58, 42]",NaN,"{""dropout"": 0.0721767197541437, ""ff_multiplier...",COMPLETE
12,12,7.372818,2026-11-01 17:31:22.031710,2026-11-01 17:34:30.558988,0 days 00:03:08.527278,0.083111,3,8,3,0.001414,2,64.0,0.000021,32,"[54, 62, 100]",NaN,"{""dropout"": 0.08311051736842226, ""ff_multiplie...",COMPLETE
29,29,7.385602,2026-11-01 18:18:58.546625,2026-11-01 18:21:42.375621,0 days 00:02:43.828996,0.061296,3,8,3,0.002593,2,32.0,0.000398,32,"[38, 58, 42]",NaN,"{""dropout"": 0.06129601849354262, ""ff_multiplie...",COMPLETE
32,32,7.405032,2026-11-01 18:27:53.194616,2026-11-01 18:31:36.008250,0 days 00:03:42.813634,0.081883,3,8,3,0.002658,2,32.0,0.000438,32,"[51, 39, 42]",NaN,"{""dropout"": 0.08188287544105287, ""ff_multiplie...",COMPLETE
24,24,7.439084,2026-11-01 18:08:48.031177,2026-11-01 18:11:16.074700,0 days 00:02:28.043523,0.030842,3,8,3,0.002041,2,64.0,0.000033,32,"[33, 40, 40]",NaN,"{""dropout"": 0.030841908048557222, ""ff_multipli...",COMPLETE
11,11,7.458770,2026-11-01 17:28:38.382670,2026-11-01 17:31:21.970971,0 days 00:02:43.588301,0.187490,3,8,3,0.001685,2,64.0,0.000013,32,"[54, 66, 59]",NaN,"{""dropout"": 0.1874900961493232, ""ff_multiplier...",COMPLETE
36,36,7.485977,2026-11-01 18:39:57.311144,2026-11-01 18:42:24.500858,0 days 00:02:27.189714,0.043934,3,8,3,0.002447,1,16.0,0.000057,32,"[27, 51, 37]",NaN,"{""dropout"": 0.04393365350110312, ""ff_multiplie...",COMPLETE
13,13,7.523233,2026-11-01 17:34:30.668954,2026-11-01 17:37:00.270410,0 days 00:02:29.601456,0.157829,2,8,3,0.002261,2,64.0,0.000554,32,"[37, 83, 40]",NaN,"{""dropout"": 0.15782875174612598, ""ff_multiplie...",COMPLETE
31,31,7.541996,2026-11-01 18:24:20.277413,2026-11-01 18:27:53.063785,0 days 00:03:32.786372,0.115523,3,8,3,0.001750,2,32.0,0.000883,32,"[46, 62, 42]",NaN,"{""dropout"": 0.1155226686711934, ""ff_multiplier...",COMPLETE


## 8. Confirm the best configurations: five splits, three seeds

Evaluate the best TOP_K completed configurations (default three; optionally five) on all five
splits with seeds 42, 43 and 44. Select the configuration with the lowest mean pipeline MSE
across these 15 runs, not the seed with the lowest score. Standard deviations describe variability.
These scores guide model selection; because search and confirmation reuse validation patients,
they are not an independent, unbiased estimate of the selected model's test performance.

In [14]:
candidates = sorted(
    (trial for trial in study.trials if trial.state == optuna.trial.TrialState.COMPLETE),
    key=lambda trial: trial.value,
)[:TOP_K]
confirmation = []
for candidate in candidates:
    print(f"Confirming configuration from trial {candidate.number}")
    result = evaluate_config(candidate.params, prepared_folds, SEEDS)
    confirmation.append(result.assign(trial=candidate.number))
confirmation = pd.concat(confirmation, ignore_index=True)
confirmation_summary = confirmation.groupby("trial").agg(
    MSE=("Encoder_HistGB", "mean"), std=("Encoder_HistGB", lambda s: s.std(ddof=0)),
    HistGB=("HistGB", "mean"), delta_MSE=("delta_MSE", "mean"),
).sort_values("MSE")
winning_trial = int(confirmation_summary.index[0])
winning_params = next(trial.params for trial in candidates if trial.number == winning_trial)
scores = confirmation.loc[confirmation["trial"].eq(winning_trial)].copy()
print("Winning encoder parameters:", winning_params)
confirmation_summary

Confirming configuration from trial 19
Fold 1/5, seed 42: MSE 7.5995
Fold 1/5, seed 43: MSE 7.1951
Fold 1/5, seed 44: MSE 7.3864
Fold 2/5, seed 42: MSE 7.5481
Fold 2/5, seed 43: MSE 7.1661
Fold 2/5, seed 44: MSE 7.5765
Fold 3/5, seed 42: MSE 6.8773
Fold 3/5, seed 43: MSE 7.7778
Fold 3/5, seed 44: MSE 8.2516
Fold 4/5, seed 42: MSE 7.0266
Fold 4/5, seed 43: MSE 7.8758
Fold 4/5, seed 44: MSE 7.2814
Fold 5/5, seed 42: MSE 7.5157
Fold 5/5, seed 43: MSE 7.6556
Fold 5/5, seed 44: MSE 7.4228
Confirming configuration from trial 27
Fold 1/5, seed 42: MSE 7.7589
Fold 1/5, seed 43: MSE 7.7295
Fold 1/5, seed 44: MSE 7.3925
Fold 2/5, seed 42: MSE 7.0377
Fold 2/5, seed 43: MSE 7.6404
Fold 2/5, seed 44: MSE 7.3738
Fold 3/5, seed 42: MSE 7.2499
Fold 3/5, seed 43: MSE 7.1947
Fold 3/5, seed 44: MSE 7.4126
Fold 4/5, seed 42: MSE 7.4565
Fold 4/5, seed 43: MSE 7.5544
Fold 4/5, seed 44: MSE 7.9101
Fold 5/5, seed 42: MSE 7.8655
Fold 5/5, seed 43: MSE 7.6440
Fold 5/5, seed 44: MSE 8.0869
Winning encoder parame

,MSE,std,HistGB,delta_MSE
trial,,,,
19,7.477092,0.336742,11.923423,-4.446331
27,7.553838,0.279559,11.923423,-4.369585


In [15]:
summary = scores[["HistGB", "Transformer", "Encoder_HistGB"]].agg(
    ["mean", lambda s: s.std(ddof=0)]
).T
summary.columns = ["MSE", "std"]
summary["RMSE"] = np.sqrt(summary["MSE"])
summary["gain_MSE_pct"] = 100 * (1 - summary["MSE"] / summary.loc["HistGB", "MSE"])
summary

,MSE,std,RMSE,gain_MSE_pct
HistGB,11.923423,0.352683,3.453031,0.000000
Transformer,8.429465,0.794165,2.903354,29.303310
Encoder_HistGB,7.477092,0.336742,2.734427,37.290727


## 9. Optional final training and test predictions

RUN_FINAL controls submission generation. The winning configuration is refitted on all training
patients with the fixed FINAL_SEED, retaining internal epoch selection and full-data refitting.
Only the encoder configuration changes; HistGB keeps its original parameters. Test patients
never fit preprocessing or either model. The optimized submission has a separate filename.

In [17]:
RUN_FINAL = True
FINAL_SEED = 43

if RUN_FINAL:
    X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
    with threadpool_limits(limits=3):
        final_prep = clone(baseline[:-1])
        train = final_prep.fit_transform(X, y)
        test = final_prep.transform(X_test)
        imputed_train = final_prep[:2].transform(X)
        imputed_test = final_prep[:2].transform(X_test)
        final_encoder = fit_encoder(X, y, imputed_train, FINAL_SEED, winning_params)
        z_train, _ = encoder_outputs(final_encoder, X, imputed_train)
        z_test, _ = encoder_outputs(final_encoder, X_test, imputed_test)
        final_histgb = clone(baseline[-1]).fit(train.join(z_train), y)
        prediction = final_histgb.predict(test.join(z_test))
    assert np.isfinite(prediction).all()
    submission = pd.DataFrame({"Index": X_test["Index"], "target": prediction})
    submission_path = DATA_DIR / "submission_encoder_histgb_optimized.csv"
    submission.to_csv(submission_path, index=False)
    print(f"Saved {submission_path} ({len(submission):,} visits)")

Saved /Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/Challenge_Parkinson_disease/data/submission_encoder_histgb_optimized.csv (23,672 visits)


## Alternatives to test later

- Add the existing engineered features to encoder inputs.
- Replace the Transformer with another encoder architecture.
- Predict ON/OFF measurements with this encoder or a separate encoder.
- Feed HistGB the encoder's final prediction instead of, or alongside, its embeddings.